# Archival Accession Processing Tool — synthetic Colab smoke run
Click **Runtime → Run all**. CPU runtimes run the offline contract check and download a report without installing packages. For one small model, first select **Runtime → Change runtime type → GPU** (free availability varies).

Synthetic inputs only. No original accession data, Drive mount, paid APIs or Codex credentials are used. Stub success is not model performance or archival quality. Vision is optional and disabled by default. GitHub is the code source: the chosen ref resolves to a recorded exact commit before execution.


In [ ]:
# Runtime check happens before cloning or installing anything.
import json, platform
from pathlib import Path
RUNTIME = {"python": platform.python_version(), "gpu_available": False, "gpu": None}
try:
    import torch
    RUNTIME["torch"] = torch.__version__
    RUNTIME["gpu_available"] = torch.cuda.is_available()
    if RUNTIME["gpu_available"]:
        RUNTIME["gpu"] = torch.cuda.get_device_name(0)
except Exception as exc:
    RUNTIME["gpu_probe_error"] = type(exc).__name__ + ": " + str(exc)
print(json.dumps(RUNTIME, indent=2))
if not RUNTIME["gpu_available"]:
    print("CPU mode: offline checks and report only; no dependency installation.")


## Run and automatically download the report
The report includes the code commit, runtime, isolated package versions/check, synthetic results, failures and skipped stages. Model setup uses a private virtual environment, so Colab's installed Gradio, Diffusers and Hugging Face packages are untouched. The first GPU setup may download a separate PyTorch/CUDA stack and take several minutes.

Optional controls below use Colab form fields; no code editing is required. For repeat comparisons, paste the report's commit into Code ref. Each run gets a fresh directory and preserves earlier work.


In [ ]:
CODE_REF = "gauri/colab-runtime-readiness-2026-10-13" # @param {type:"string"}
ENABLE_MODELS = True # @param {type:"boolean"}
ENABLE_VISION = False # @param {type:"boolean"}
import subprocess, sys, tempfile, zipfile
from datetime import datetime, timezone
BASE = Path(tempfile.mkdtemp(prefix="archival-smoke-", dir="/content"))
CHECKOUT = BASE / "code"
OUT = BASE / "reports"
OUT.mkdir()
BUNDLE = OUT / "archival-colab-report.zip"
try:
    subprocess.run(["git", "clone", "--quiet", "https://github.com/terbe2022/RIMS-Archival-Project-.git", str(CHECKOUT)], check=True)
    subprocess.run(["git", "checkout", "--detach", CODE_REF], cwd=CHECKOUT, check=True)
    COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, text=True).strip()
    print("Code commit:", COMMIT)
    # Execute the helper from that checkout, even when rerunning this cell.
    import importlib.util
    spec = importlib.util.spec_from_file_location("colab_runtime", CHECKOUT / "validation/colab_runtime.py")
    helper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(helper)
    report, BUNDLE = helper.run_session(OUT, runtime=RUNTIME, enable_models=ENABLE_MODELS,
        vision_models=["HuggingFaceTB/SmolVLM-256M-Instruct"] if ENABLE_VISION else None)
    print(json.dumps(report["stages"], indent=2))
    if report.get("error"):
        print(report["error"])
except Exception as exc:
    failure = {"runtime": RUNTIME, "requested_code_ref": CODE_REF,
               "created_utc": datetime.now(timezone.utc).isoformat(),
               "status": "bootstrap_failed", "error": type(exc).__name__ + ": " + str(exc)}
    path = OUT / "diagnostics.json"
    path.write_text(json.dumps(failure, indent=2), encoding="utf-8")
    with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(path, path.name)
    print("Setup failed; diagnostic report still available.")
finally:
    from google.colab import files
    print("Report:", BUNDLE)
    try:
        files.download(str(BUNDLE))
    except Exception:
        from IPython.display import FileLink, display
        display(FileLink(str(BUNDLE)))
        print("If the browser blocks download, use the Files sidebar to download the ZIP shown above.")


## Codex CLI investigation (not part of Run all)
ChatGPT-account device authentication is documented for Codex CLI, but has **not been tested in this Colab runtime**. Browser access does not itself provide direct runtime access. See [setup and verification status](https://github.com/terbe2022/RIMS-Archival-Project-/blob/gauri/colab-runtime-readiness-2026-10-13/docs/setup/colab-setup.md).

Do not run login in notebook cells: device codes/output can be saved with the notebook. Authentication must happen in a private terminal, with credentials outside the repository and reports. Do not upload an existing authentication cache. Code changes return through a GitHub PR; never treat a temporary Colab edit as the reviewed source.
